<a href="https://colab.research.google.com/github/korukondarrao/LIAR_weightclassification/blob/main/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP ONE: Handling the Dataset


Dependencies and Libraries

In [ ]:
# Install core libraries: transformers UPGRADED (model and tokenizer), datasets (data), scikit-learn (baseline stuff and metrics)
!pip install -U transformers
!pip install datasets scikit-learn

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [ ]:
# Discrepency between evaluation_strategy and save_strategy because outdated HuggingFace Transformers, so upgraded transformers with -U above
# 5.5.4, which is greater than 4.0!
import transformers
print(transformers.__version__)

Dataset Loading from Drive

In [ ]:
# Mount from Drive
from google.colab import drive
drive.mount('/content/drive')

# Set a base path connection, so all this doesn't need to be typed EACH time
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Final_Project/"
RESULTS_PATH = BASE_PATH + "results/"
MODEL_PATH = BASE_PATH + "models/"

os.makedirs(RESULTS_PATH, exist_ok = True)
os.makedirs(MODEL_PATH, exist_ok = True)

In [ ]:
# Separate columns based on features
COLUMNS = ["id", "label", "statement", "subject", "speaker", "job", "state", "party", "barely_true", "false", "half_true", "mostly_true", "pants_fire", "context"]

# .tsv files read, NOT .csv, so sep = "\t"
def load_liar(path):
    return pd.read_csv(path, sep = "\t", header = None, names = COLUMNS)

# Upload .tsv files for preprocessing
train_df = load_liar(BASE_PATH + "train.tsv")
val_df   = load_liar(BASE_PATH + "valid.tsv")
test_df  = load_liar(BASE_PATH + "test.tsv")

Preprocessing

In [ ]:
# Six features for truthfulness (0-5)
label_map = {"pants-fire": 0, "false": 1, "barely-true": 2, "half-true": 3, "mostly-true": 4, "true": 5}

# Preprocess each file: drop missing statements, 1-hot-encoding for "false" categorical variables, strip whitespace (but keep text the same)
def preprocess(df):
  df = df.copy()
  df = df.dropna(subset=["statement"])
  df["statement"] = df["statement"].str.strip()
  df["label"] = df["label"].map(label_map)
  return df[["statement", "label"]]

train_df = preprocess(train_df)
val_df   = preprocess(val_df)
test_df  = preprocess(test_df)

# Only Text (input) and Label (output, which is specified above) kept for current relevancy
train_df = train_df.rename(columns = {"statement": "text"})
val_df   = val_df.rename(columns = {"statement": "text"})
test_df  = test_df.rename(columns = {"statement": "text"})

In [ ]:
# LIAR imbalances mean some classes are rarer, so value_counts for how many examples per layer

train_df["label"].value_counts().sort_index().plot(kind = "bar")
plt.title("Label Distribution")
plt.savefig(RESULTS_PATH + "label_dist.png")
plt.show()

# Results show WAY less for pants-fire

# STEP TWO: The Baseline Model

In [ ]:
# TF-IDF for word2vec conversion: bases on importance of word frequency
from sklearn.feature_extraction.text import TfidfVectorizer

# Logistic Regression as baseline for above features
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features = 5000)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])

y_train = train_df["label"]
y_val = val_df["label"]

clf = LogisticRegression(max_iter = 1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)

baseline_acc = accuracy_score(y_val, y_pred)
baseline_f1 = f1_score(y_val, y_pred, average = 'macro')

print("Baseline Accuracy:", baseline_acc)
print("Baseline F1 Score:", baseline_f1)

# RESULTS
# Baseline Accuracy: 0.23909657320872274
# Baseline F1 Score: 0.22423559784045832

Confusion Matrix

In [ ]:
# Analysis of true positives, false negatives, true negatives, and false positives

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_pred)

labels = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]

# Normalize to understand ACTUAL proportions (class imbalances)
cm_norm = cm.astype('float') / cm.sum(axis = 1, keepdims = True)

# Plot heatmap/confusion matrix w/ blue gradient (best color, duh)
plt.figure(figsize = (7, 6))
plt.imshow(cm_norm, cmap = "Blues")
plt.title("Normalized Confusion Matrix")
plt.colorbar()

plt.xticks(np.arange(6), labels, rotation = 45)
plt.yticks(np.arange(6), labels)

# Print percentages for each box in matrix (both raw counts AND percentages)
for i in range(cm.shape[0]):
  for j in range(cm.shape[1]):
    plt.text(j, i, f"{cm[i, j]}\n({cm_norm[i, j]:.2f})", ha = "center", va = "center", color = "black")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()

plt.savefig(RESULTS_PATH + "baseline_cm.png")
plt.show()

# RESULTS
# Accuracies Per Class: Pants-Fire (10%), False (35%), Barely-True (15%), Half-True (29%), Mostly-True (24%), True (21%)
# Highest Prediction Attributed Per Class: Pants-Fire (30% for False), False (35% for False) Barely-True (28% for False), Half-True (29% for Half-True), Mostly-True (26% for Half-True), True (27% for Half-True)
# Barely predicts pants-fire (rare class) and vice versa for frequent classes (false and half-true), predictions mainly false, half-true, or mostly-true
# Class imbalance significantly affects performance -- which is why class-weighted loss can prove very useful.

In [ ]:
# Save results in a .txt file for later comparison
with open(RESULTS_PATH + "baseline.txt", "w") as f:
  f.write(f"Accuracy: {baseline_acc}\nF1 Score: {baseline_f1}")

# STEP THREE: RoBERTa Model


Tokenization Process

In [ ]:
# Pull up dataset for training model
from datasets import Dataset

# Convert pandas DataFrame to HuggingFace dataset format for transformers' Trainer format
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Convert text to inputs for RoBERTa model (words --> tokens, tokens --> IDs, and padding/truncation)
def tokenize(batch):
  return tokenizer(batch["text"], padding = "max_length", truncation = True, max_length = 128)

# Train based on above
train_dataset = train_dataset.map(tokenize, batched = True)
val_dataset = val_dataset.map(tokenize, batched = True)

train_dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "label"])
val_dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "label"])

In [ ]:
# Fine-tuning the pre-trained model
from transformers import AutoModelForSequenceClassification

# Load pre-trained RoBERTa with classification head and the 6 output labels
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels = 6)

# RESULTS
# RobertaForSequenceClassification LOAD REPORT from: roberta-base
# Key                             | Status     |
# --------------------------------+------------+-
# lm_head.layer_norm.bias         | UNEXPECTED |
# lm_head.dense.bias              | UNEXPECTED |
# lm_head.layer_norm.weight       | UNEXPECTED |
# lm_head.bias                    | UNEXPECTED |
# roberta.embeddings.position_ids | UNEXPECTED |
# lm_head.dense.weight            | UNEXPECTED |
# classifier.dense.weight         | MISSING    |
# classifier.out_proj.bias        | MISSING    |
# classifier.out_proj.weight      | MISSING    |
# classifier.dense.bias           | MISSING    |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# - MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

In [ ]:
# Function for the metrics
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis = 1)

  # Macro-F1 better for class imbalance than accuracy
  acc = accuracy_score(labels, preds)
  f1 = f1_score(labels, preds, average = 'macro')

  return {"accuracy": acc, "macro_f1": f1}

Training

In [ ]:
# Error with parameter names, so print names to check
import transformers
from transformers import TrainingArguments

print(transformers.__version__)
print(TrainingArguments.__init__.__code__.co_varnames)

In [ ]:
from transformers import Trainer, TrainingArguments

# Forward pass for predictions, compute loss, backprop, and update model's weights -- REPEAT over epochs
training_args = TrainingArguments(output_dir = MODEL_PATH, num_train_epochs = 2, per_device_train_batch_size = 16, per_device_eval_batch_size = 16, eval_strategy = "epoch", save_strategy = "epoch", logging_dir = RESULTS_PATH)

# Subset with select(range(5000)) for faster iteration but progress depictions
trainer = Trainer(model = model, args = training_args, train_dataset = train_dataset.select(range(5000)), eval_dataset = val_dataset.select(range(1000)), compute_metrics = compute_metrics)

trainer.train()

# RESULTS
# Epoch	Training Loss	Validation Loss	Accuracy	Macro F1
# 1	No log	1.765655	0.210000	0.086462
# 2	1.739255	1.706569	0.263000	0.144355
# TrainOutput(global_step=626, training_loss=1.728209717966878, metrics={'train_runtime': 248.9603, 'train_samples_per_second': 40.167, 'train_steps_per_second': 2.514, 'total_flos': 657801262080000.0, 'train_loss': 1.728209717966878, 'epoch': 2.0})

# Validation loss decreases from 1.765655 to 1.706569 (model seems to learn generalized patterns better after training).
# Accuracy from 21% to 26.3%, showing improvement over epochs, but weak performance overall.
# Macro F1 (model performance across all classes equally) increases from 0.086462 to 0.144355, still showing strong imbalance among predictions.
# Despite accuracy improvement, model is still failing to learn balanced representations across the 6 LIAR features (biased towards frequent labels and struggles with fine-grained distinctions).
# Roberta F1 lower than baseline, probably due to mild overfitting of dominant classes; thus, class weights should be addressed for these metrics, specifically.

# STEP FOUR: New RoBERTa Model (Classification Weights)